# 82514 · Sesión S15 — La dinámica del brazo y el par calculado

**Bloque 5** · lunes 9 de noviembre de 2026 · 2 h · IQS Universitat Ramon Llull

El 2R del bloque con sus dos motores: M(q), c y g, la caída libre y tres controladores sobre el giro de S12 (PD, PD + g y par calculado), con el modelo exacto y equivocado. No depende del S15a: define sus propios datos.

| | |
|---|---|
| **En clase** | Abridlo al acabar el taller S15a y ejecutad la instalación. Sección 1, sin deteneros, y sección 3 (cuatro minutos); se pone en común al acabar. |
| **En casa** | Secciones 2 y 4 y ejercicios 1 a 3; las soluciones, al final. |
| **Entrega** | No se sube. |

Cada paso tiene tres celdas: **▶ Parámetros**, lo único que se cambia; **⚙ Cálculo**, que no hace falta tocar; y **✔ Resultado**, lo que sale.

---

In [ ]:
# @title ⚙ Instalación · ejecutad esta celda primero
# Ejecutar una sola vez. En Colab tarda unos segundos.
import importlib, subprocess, sys

def asegurar(mods):
    faltan = []
    for pip_name, import_name in mods:
        try:
            importlib.import_module(import_name)
        except ImportError:
            faltan.append(pip_name)
    if faltan:
        print('Instalando:', ' '.join(faltan))
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + faltan, check=False)
    else:
        print('Todo instalado ya.')

asegurar([('numpy', 'numpy'), ('scipy', 'scipy'), ('matplotlib', 'matplotlib'), ('control==0.10.2', 'control')])

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import control as ct
from scipy.integrate import solve_ivp

np.set_printoptions(precision=4, suppress=True)
plt.rcParams['figure.figsize'] = (9, 3.4)
plt.rcParams['axes.grid'] = True
IQS_AZUL, IQS_VERDE = '#1B2A80', '#1FA355'
GRIS, AMBAR, NAVY, ROJO = '#8A8FA8', '#B9770E', '#2E3FA3', '#C0392B'
COMA = FuncFormatter(lambda v, p: f'{v:g}'.replace('.', ','))   # ejes con coma decimal

def ejes(*axs):
    for ax in axs:
        ax.xaxis.set_major_formatter(COMA); ax.yaxis.set_major_formatter(COMA)

def coma(x, fmt='.3g'):
    """Un número con coma decimal, para los rótulos de las figuras."""
    return format(x, fmt).replace('.', ',')

print('python-control', ct.__version__, '- listo.')
if ct.__version__ != '0.10.2':
    print('AVISO: este cuaderno se probó con python-control 0.10.2. Ejecuta  !pip install -q control==0.10.2  y reinicia el entorno.')

## 1. La ecuación del brazo

*En clase, sin deteneros · diapositivas 52 y 53*

El 2R en vertical, con un motor de S7 y G = 50 en cada articulación:

[M(q) + 0,25·I]·q̈ + c(q, q̇) + 3,125·q̇ + g(q) = 1,25·V

La deducción, paso a paso, está en la celda ⚙. Mirad cómo cambia M(q) con la postura.

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# el 2R de B4: longitudes de los eslabones [m]
A1 = 0.35  # @param {type:"number"}
A2 = 0.2  # @param {type:"number"}
# masas puntuales en el codo (motor y eslabón) y en la mano [kg], como en S13
M1 = 1.0  # @param {type:"number"}
M2 = 1.0  # @param {type:"number"}
# gravedad [m/s²]
GRAV = 9.81  # @param {type:"number"}
# el motor de S7: tensión máxima [V], resistencia [Ω], constante [N·m/A], inercia del rotor [kg·m²]
V_BUS = 24.0  # @param {type:"number"}
R = 2.0  # @param {type:"number"}
K = 0.05  # @param {type:"number"}
JM = 1e-4  # @param {type:"number"}
# reductora, la misma en las dos articulaciones [-]
G = 50  # @param {type:"number"}
# posturas que se tabulan, (q1, q2) [°]
POSTURAS = [[0, 0], [0, 90], [30, 60], [0, 180]]  # @param {type:"raw"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
# Euler-Lagrange, paso a paso (ángulos desde la horizontal; q2, el del codo, relativo al eslabón 1):
#  1. Posiciones: codo p1 = a1·(cos q1, sen q1); mano p2 = p1 + a2·(cos(q1 + q2), sen(q1 + q2)).
#  2. Velocidades: |v1|² = a1²·q1'²;  |v2|² = a1²·q1'² + a2²·(q1' + q2')² + 2·a1·a2·cos q2·q1'·(q1' + q2').
#  3. Energías: Ec = ½·m1·|v1|² + ½·m2·|v2|² = ½·q'ᵀ·M(q)·q'  (-> Mq);
#     Ep = m1·g·a1·sen q1 + m2·g·(a1·sen q1 + a2·sen(q1 + q2)), y g(q) = ∂Ep/∂q  (-> gq).
#  4. Con L = Ec - Ep, d/dt(∂L/∂q') - ∂L/∂q = M(q)·q'' + c(q, q') + g(q) = G·K·i, el par de los motores.
#     c, Coriolis (con q1'·q2') y centrífugos (con q1'² o q2'²), sale de que M cambia con q2  (-> cq).
#  Los motores, en tensión: G·K·i = 1.25·V - 3.125·q', y cada rotor suma G²·Jm = 0.25 a la diagonal:
#     [M(q) + 0.25·I]·q'' + c + 3.125·q' + g = 1.25·V   (codo estirado y quieto: la ecuación de S14).
#  Se desprecia que el rotor del codo gira también con el eslabón 1 (Jm en M11 y G·Jm en M12).
KTAU, BEQ, JR = G*K/R, G**2*K**2/R, G**2*JM             # 1.25 N·m/V, 3.125 N·m·s/rad y 0.25 kg·m²
H2 = M2*A1*A2                                            # 0.07

def Mq(q):
    """La inercia del brazo, M(q), sin los rotores: [[0.285 + 0.14·cos q2, 0.04 + 0.07·cos q2], [·, 0.04]]."""
    c2 = np.cos(q[1])
    return np.array([[M1*A1**2 + M2*(A1**2 + A2**2 + 2*A1*A2*c2), M2*(A2**2 + A1*A2*c2)],
                     [M2*(A2**2 + A1*A2*c2),                      M2*A2**2]])

def cq(q, dq):
    """c(q, q') = 0.07·sen q2·(-2·q1'·q2' - q2'²; q1'²): Coriolis (con q1'·q2') y centrífugos (con q1'² o q2'²)."""
    s2 = np.sin(q[1])
    return np.array([-H2*s2*(2*dq[0]*dq[1] + dq[1]**2), H2*s2*dq[0]**2])

def gq(q):
    """g(q) = 9.81·(0.7·cos q1 + 0.2·cos(q1 + q2); 0.2·cos(q1 + q2)), en N·m."""
    return GRAV*np.array([(M1 + M2)*A1*np.cos(q[0]) + M2*A2*np.cos(q[0] + q[1]), M2*A2*np.cos(q[0] + q[1])])

# cada postura: M, sus autovalores, M + 0.25·I y g
TABLA_M = [(grados, Mq(np.radians(grados)), np.linalg.eigvalsh(Mq(np.radians(grados))),
            Mq(np.radians(grados)) + JR*np.eye(2), gq(np.radians(grados))) for grados in POSTURAS]
M_HOMBRO, G_HOMBRO = Mq([0, 0])[0, 0] + JR, gq([0, 0])[0]   # el hombro, con el codo estirado y quieto
GJM = G*JM                                               # el rotor del codo con el eslabón 1, que se desprecia
GJM_PCT = 100*G*JM/Mq([0, 0])[0, 1]                      # en % de M12 estirado

In [ ]:
# @title ✔ Resultado
print(f'tau = {KTAU} N·m por voltio ({KTAU*V_BUS:.0f} N·m con 24 V); contratensión {BEQ:.3f} N·m·s/rad; rotor {JR} kg·m²\n')
print(f"{'(q1, q2)':>12} {'M11':>6} {'M12':>6} {'M22':>6} {'autovalores de M':>17} {'M11 + 0.25':>11} {'M22 + 0.25':>11} {'g1 [N·m]':>9} {'g2 [N·m]':>9}")
for grados, M, ev, Mr, g in TABLA_M:
    print(f"{'(' + str(grados[0]) + '°, ' + str(grados[1]) + '°)':>12} {M[0, 0]:6.3f} {M[0, 1]:6.3f} {M[1, 1]:6.3f}"
          f" {ev[0]:8.4f} {ev[1]:8.4f} {Mr[0, 0]:11.3f} {Mr[1, 1]:11.3f} {g[0]:9.3f} {g[1]:9.3f}")
print(f"\nhombro estirado y codo quieto: {M_HOMBRO:.3f}·q1'' + {BEQ:.3f}·q1' + {G_HOMBRO:.3f}·cos q1 = {KTAU}·V   (la ecuación de S14)")
print(f'el rotor del codo con el eslabón 1, que se desprecia: G·Jm = {GJM:.3f}, un {GJM_PCT:.1f} % de M12 estirado')

**Qué tiene que salir:** estirado (0°, 0°), M = [[0,425; 0,11]; [0,11; 0,04]] y g = (8,83; 1,96) N·m; plegado (0°, 180°), M₁₁ = 0,145. Con los rotores, el hombro estirado da 0,675·q̈₁ + 3,125·q̇₁ + 8,83·cos q₁ = 1,25·V, la ecuación de S14.

### La energía, con el jacobiano de S11

M es la suma de m·Jᵀ·J, con el jacobiano de cada masa: las dos formas de la energía cinética tienen que coincidir. Cambiad la postura y las velocidades.

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# una postura cualquiera, (q1, q2) [rad]
Q_PRUEBA = [0.4, 1.1]  # @param {type:"raw"}
# unas velocidades cualesquiera, (q1', q2') [rad/s]
DQ_PRUEBA = [0.7, -0.3]  # @param {type:"raw"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
def jac_masa(q, cual):
    """Jacobiano de la posición de la masa del codo (cual = 1) o de la mano (cual = 2): el de S11."""
    s1, c1, s12, c12 = np.sin(q[0]), np.cos(q[0]), np.sin(q[0] + q[1]), np.cos(q[0] + q[1])
    if cual == 1:
        return np.array([[-A1*s1, 0.0], [A1*c1, 0.0]])
    return np.array([[-A1*s1 - A2*s12, -A2*s12], [A1*c1 + A2*c12, A2*c12]])

qp, dqp = np.array(Q_PRUEBA), np.array(DQ_PRUEBA)
EC_M = 0.5*dqp @ Mq(qp) @ dqp                                                              # ½·q'ᵀ·M·q'
EC_J = 0.5*(M1*np.sum((jac_masa(qp, 1) @ dqp)**2) + M2*np.sum((jac_masa(qp, 2) @ dqp)**2))  # ½·Σ m·|J·q'|²
MJ = M1*jac_masa(qp, 1).T @ jac_masa(qp, 1) + M2*jac_masa(qp, 2).T @ jac_masa(qp, 2)        # Σ m·Jᵀ·J
DIF_MJ = np.abs(Mq(qp) - MJ).max()

In [ ]:
# @title ✔ Resultado
print(f"½·q'ᵀ·M·q'      = {EC_M:.6f} J")
print(f"½·Σ m·|J·q'|²  = {EC_J:.6f} J")
print('M - Σ m·Jᵀ·J =', DIF_MJ, '(lo mismo)')

## 2. Soltar el brazo

*En casa*

Se despeja q̈ = [M(q) + 0,25·I]⁻¹·(τ − c − 3,125·q̇ − g) y `solve_ivp` integra el brazo soltado parado y estirado en horizontal, sin motores y con los motores a 0 V. Mirad la energía: sin motores se conserva; con ellos, la contratensión la disipa.

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# duración de la simulación [s]
T_CAIDA = 4.0  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
def caida(con_motores, T):
    """El brazo soltado parado y estirado en horizontal, sin motores o con los motores a 0 V."""
    Mr = JR*np.eye(2) if con_motores else np.zeros((2, 2))
    b = BEQ if con_motores else 0.0
    def f(t, x):
        q, dq = x[:2], x[2:]
        return np.r_[dq, np.linalg.solve(Mq(q) + Mr, -cq(q, dq) - b*dq - gq(q))]     # tau = 0
    sol = solve_ivp(f, (0, T), np.zeros(4), rtol=1e-10, atol=1e-12, max_step=1e-3, dense_output=True)
    t = np.linspace(0, T, 4001); X = sol.sol(t)
    Ep = GRAV*(M1*A1*np.sin(X[0]) + M2*(A1*np.sin(X[0]) + A2*np.sin(X[0] + X[1])))
    Ec = np.array([0.5*X[2:, i] @ (Mq(X[:2, i]) + Mr) @ X[2:, i] for i in range(len(t))])
    return t, X, Ec + Ep

CAIDAS = {con: caida(con, T_CAIDA) for con in (False, True)}    # False: sin motores; True: motores a 0 V

In [ ]:
# @title ✔ Resultado
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, con, titulo in ((axs[0], False, 'Sin motores: la energía se conserva'), (axs[1], True, 'Con los motores a 0 V: frenan')):
    t, X, E = CAIDAS[con]
    print(f'{titulo}: energía de {E[0]:.3f} a {E[-1]:.3f} J' + ('' if con else f' (deriva {E[-1] - E[0]:.1e} J)'))
    print(f'   hombro: mínimo {np.degrees(X[0]).min():.1f}°, al final {np.degrees(X[0, -1]):.1f}°; '
          f'codo al final {np.degrees(X[1, -1]):.1f}°; pico de velocidad del hombro {np.degrees(np.abs(X[2]).max()):.1f} °/s')
    ax.plot(t, np.degrees(X[0]), color=IQS_AZUL, lw=2.2, label=r'$q_1$, hombro')
    ax.plot(t, np.degrees(X[1]), color=IQS_VERDE, lw=2.2, label=r'$q_2$, codo')
    ax.set_title(titulo); ax.set_xlabel('t [s]'); ax.set_ylabel('ángulo [°]'); ax.set_xlim(0, T_CAIDA)
    ax.legend(loc='lower left', fontsize=8.5)
ejes(*axs); plt.tight_layout(); plt.show()

## 3. Tres controladores

*En clase · diapositivas 54 a 58; se pone en común al acabar*

PD por articulación, PD + ĝ y par calculado sobre el giro de S12, de (30°, 60°) a (1,0°; 74,5°), en 2 s y cuatro veces más rápido, en 0,5 s. Mirad el error del hombro.

### Qué pesa cada término

El máximo de cada término de la ecuación en el hombro, a lo largo del giro (diapositiva 54).

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# el giro de S12, (q1, q2) al empezar y al acabar [°]
Q0_GRADOS = [30, 60]  # @param {type:"raw"}
QF_GRADOS = [1.0, 74.5]  # @param {type:"raw"}
# duraciones del giro: el de S12 y cuatro veces más rápido [s]
T_GIROS = [2.0, 0.5]  # @param {type:"raw"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
Q0, QF = np.radians(Q0_GRADOS), np.radians(QF_GRADOS)

def quintico(t, T, q0, qf):
    """La trayectoria de quinto grado de S12: posición, velocidad y aceleración en t; parada después de T."""
    s = min(max(t/T, 0.0), 1.0)
    p = 10*s**3 - 15*s**4 + 6*s**5
    dp = (30*s**2 - 60*s**3 + 30*s**4)/T if t <= T else 0.0
    ddp = (60*s - 180*s**2 + 120*s**3)/T**2 if t <= T else 0.0
    d = qf - q0
    return q0 + p*d, dp*d, ddp*d

def pesos(T):
    """El máximo, a lo largo del giro, de cada término en el hombro: inercia, rotor, c, contratensión y gravedad."""
    filas = []
    for t in np.linspace(0, T, 4001):
        q, dq, ddq = quintico(t, T, Q0, QF)
        filas.append([(Mq(q) @ ddq)[0], JR*ddq[0], cq(q, dq)[0], BEQ*dq[0], gq(q)[0]])
    return np.abs(np.array(filas)).max(axis=0)

PESOS = {T: pesos(T) for T in T_GIROS}

In [ ]:
# @title ✔ Resultado
print(f"{'en el hombro, máximo [N·m]':>28} {'inercia':>8} {'rotor':>7} {'c':>7} {'contratensión':>14} {'gravedad':>9}")
for T, mx in PESOS.items():
    print(f"{'giro en ' + str(T) + ' s':>28} {mx[0]:8.3f} {mx[1]:7.3f} {mx[2]:7.3f} {mx[3]:14.2f} {mx[4]:9.2f}")

**Qué tiene que salir:** en 2 s, la gravedad, 7,36 N·m, frente a 0,23 + 0,18 de inercia (brazo y rotor); en 0,5 s, la inercia sube dieciséis veces, a 3,69 + 2,92, y casi la iguala.

### Los tres controladores

Los tres con ω_n = 10 rad/s y ζ = 1 (diapositivas 55 y 56): un PD por articulación con la diagonal estirada, el mismo más ĝ(q)/1,25, y el par calculado, que con el modelo exacto deja ë + 2ζω_n·ė + ω_n²·e = 0. El controlador calcula cada `H` segundos y satura a ±24 V.

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# el objetivo de los tres: frecuencia natural [rad/s] y amortiguamiento [-]
WN = 10.0  # @param {type:"number"}
ZETA = 1.0  # @param {type:"number"}
# periodo del controlador: calcula cada milisegundo, el bucle de S14 [s]
H = 1e-3  # @param {type:"number"}
# True: el D sobre el error; False: sobre la salida, -Kd·q', como el lazo de S14 [-]
D_SOBRE_ERROR = True  # @param {type:"boolean"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
Mjj = np.diag(Mq([0, 0])) + JR                           # la diagonal estirada, con los rotores: 0.675 y 0.29
KP = Mjj*WN**2/KTAU                                      # Kp = Mjj·wn²/1.25: 54 y 23.2 V/rad
KD = (2*ZETA*WN*Mjj - BEQ)/KTAU                          # Kd = (2·zeta·wn·Mjj - 3.125)/1.25: 8.30 y 2.14 V·s/rad
Me = Mq([0, 0]); Mr = Me + JR*np.eye(2)
ACOPLO = (Me[0, 1]/np.sqrt(Me[0, 0]*Me[1, 1]), Mr[0, 1]/np.sqrt(Mr[0, 0]*Mr[1, 1]))    # sin rotores y con ellos

def control(modo, t, q, dq, I, T, factor=1.0, Ki=(0.0, 0.0), d_error=True):
    """La tensión de los dos motores. modo: 'pd', 'pd+g' o 'par'. factor escala M, c y g del modelo del
    controlador (el gorro); Ki añade el término I de S14 al PD y al PD + g; I es la integral del error."""
    q_ref, dq_ref, ddq_ref = quintico(t, T, Q0, QF)
    e, de = q_ref - q, dq_ref - dq
    if modo == 'par':                                    # par calculado: tau = 1.25·V
        acel = ddq_ref + 2*ZETA*WN*de + WN**2*e
        return ((factor*Mq(q) + JR*np.eye(2)) @ acel + factor*cq(q, dq) + BEQ*dq + factor*gq(q))/KTAU
    D = KD*de if d_error else -KD*dq                     # el D, sobre el error (la trayectoria no salta) o sobre la salida
    V = KP*e + D + np.asarray(Ki)*I                      # PD (con Ki, PID) por articulación
    if modo == 'pd+g':
        V = V + factor*gq(q)/KTAU                        # más la gravedad que cree el controlador
    return V

def aceleracion(q, dq, V):
    """El brazo con sus dos motores: [M(q) + 0.25·I]·q'' = 1.25·V - c - 3.125·q' - g, con V recortada a ±24 V."""
    tau = KTAU*np.clip(V, -V_BUS, V_BUS)
    return np.linalg.solve(Mq(q) + JR*np.eye(2), tau - cq(q, dq) - BEQ*dq - gq(q))

def sim2R(modo, T, factor=1.0, Ki=(0.0, 0.0), despues=1.0, d_error=True):
    """El giro de S12 en T segundos y 'despues' segundos más. Devuelve t, el error q_ref - q en grados y V."""
    n = int(round((T + despues)/H))
    q, dq, I = Q0.copy(), np.zeros(2), np.zeros(2)
    t = np.arange(n + 1)*H; E = np.zeros((n + 1, 2)); Vs = np.zeros((n + 1, 2))
    def f(t_, q_, dq_):                                  # derivadas de q, q' y la integral del error, con V fija
        return dq_, aceleracion(q_, dq_, V), quintico(t_, T, Q0, QF)[0] - q_
    for i in range(n + 1):
        V = control(modo, t[i], q, dq, I, T, factor, Ki, d_error)   # el controlador calcula
        E[i] = quintico(t[i], T, Q0, QF)[0] - q; Vs[i] = V
        if i == n:
            break
        k1 = f(t[i], q, dq)                              # y el brazo avanza un periodo H (Runge-Kutta)
        k2 = f(t[i] + H/2, q + H/2*k1[0], dq + H/2*k1[1])
        k3 = f(t[i] + H/2, q + H/2*k2[0], dq + H/2*k2[1])
        k4 = f(t[i] + H, q + H*k3[0], dq + H*k3[1])
        q = q + H/6*(k1[0] + 2*k2[0] + 2*k3[0] + k4[0])
        dq = dq + H/6*(k1[1] + 2*k2[1] + 2*k3[1] + k4[1])
        I = I + H/6*(k1[2] + 2*k2[2] + 2*k3[2] + k4[2])
    return t, np.degrees(E), Vs

NOMBRES = {'pd': 'PD por articulación', 'pd+g': 'PD + gravedad', 'par': 'par calculado'}
SIMS = {(T, modo): sim2R(modo, T, d_error=D_SOBRE_ERROR) for T in T_GIROS for modo in NOMBRES}   # los seis casos
POLOS = ct.poles(ct.tf([1], [1, 2*ZETA*WN, WN**2]))     # el error del par calculado con el modelo exacto

In [ ]:
# @title ✔ Resultado
print(f'acoplo estirado: sin rotores {ACOPLO[0]:.3f}, con ellos {ACOPLO[1]:.3f}')
print('Kp =', KP.round(2), 'V/rad   Kd =', KD.round(2), 'V·s/rad' + ('' if D_SOBRE_ERROR else '   (D sobre la salida)'))
print(f"\n{'':22s} {'error máximo [°]':>18} {'al final [°]':>16} {'|V| máx. [V]':>14}")
print(f"{'':22s} {'hombro':>9} {'codo':>8} {'hombro':>8} {'codo':>7} {'hombro':>7} {'codo':>6}")
fig, axs = plt.subplots(1, len(T_GIROS), figsize=(11, 3.8), sharey=True, squeeze=False)
axs = axs[0]
emax = max(np.abs(SIMS[c][1][:, 0]).max() for c in SIMS)
for ax, T in zip(axs, T_GIROS):
    print(f'giro en {T} s')
    for modo, col, lw in (('pd', ROJO, 2.2), ('pd+g', AMBAR, 2.2), ('par', IQS_VERDE, 2.8)):
        t, e, V = SIMS[(T, modo)]
        em, ef, vm = np.abs(e).max(axis=0), e[-1], np.abs(V).max(axis=0)
        print(f'  {NOMBRES[modo]:20s} {em[0]:9.3f} {em[1]:8.3f} {ef[0]:8.3f} {ef[1]:7.3f} {vm[0]:7.1f} {vm[1]:6.1f}')
        ax.plot(t, e[:, 0], color=col, lw=lw, label=NOMBRES[modo])
    tope = max(7.5, 1.1*emax)
    ax.axvline(T, color=GRIS, lw=1, ls=':'); ax.text(T, 0.97*tope, ' fin del giro', color=GRIS, fontsize=9, va='top')
    ax.axhline(0, color='#999', lw=0.8); ax.set_xlim(0, T + 1.0); ax.set_ylim(-6, tope)
    ax.set_title(f'Giro de S12 en {coma(T, "g")} s'); ax.set_xlabel('t [s]')
axs[0].set_ylabel('error del hombro [°]'); axs[-1].legend(loc='lower right', fontsize=8.5)
ejes(*axs); plt.tight_layout(); plt.show()
print('polos del error del par calculado:', POLOS)

**Qué tiene que salir:** K_p = (54; 23,2) V/rad, K_d = (8,30; 2,14) V·s/rad y acoplo estirado 0,25. Error máximo del hombro: en 2 s, 6,46° (PD), 1,27° (PD + g) y 0,002° (par calculado); en 0,5 s, 6,50°, 5,15° y 0,02°. Como mucho, 9,4 V. Con `D_SOBRE_ERROR = False`, el PD en 0,5 s da 10,75° y 7,70°.

## 4. El modelo equivocado

*En casa · la tarjeta verde de la diapositiva 58*

M̂, ĉ y ĝ, lo que cree el controlador, un 20 % por debajo y por encima; el modelo del motor (0,25, 3,125 y 1,25) se da por bueno. Mirad el error al final del giro.

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# el modelo del controlador: M̂, ĉ y ĝ = factor·(M, c, g); el primero, el corto [-]
FACTORES = [0.8, 1.2]  # @param {type:"raw"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
FC = FACTORES[0]                                         # el modelo corto, para las celdas siguientes
EQUIVOCADO = {}
for T in T_GIROS:
    for factor in FACTORES:
        _, ep, _ = sim2R('par', T, factor=factor)
        _, eg, _ = sim2R('pd+g', T, factor=factor)
        # par calculado y PD + g: error máximo y al final, (hombro; codo)
        EQUIVOCADO[(T, factor)] = (np.abs(ep).max(axis=0), ep[-1], np.abs(eg).max(axis=0), eg[-1])

In [ ]:
# @title ✔ Resultado
print(f"{'':16s} {'par calculado: máximo':>23} {'al final':>16}   {'PD + g: máximo':>16} {'al final':>16}")
for (T, factor), filas in EQUIVOCADO.items():
    print(f'{T} s, x{factor}:      ' + '  '.join(f'({x[0]:6.2f}; {x[1]:5.2f})°' for x in filas))

**Qué tiene que salir:** al 80 %, al final quedan 1,68° en el hombro con el par calculado y 1,26° con el PD + g; en 0,5 s, el par calculado sigue ganando: 1,90° de máximo frente a 4,32°. Al 120 %, el par calculado da 1,48° de máximo en 0,5 s y −1,33° al final (PD + g, −1,24°).

### Por qué el par calculado deja más error parado

El 20 % de la gravedad que falta lo sostiene el error, contra la rigidez de cada lazo: 1,25·K_p en el PD + g y (0,8·M + 0,25·I)·ω_n² en el par calculado.

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
G_FALTA = (1 - FC)*gq(QF)                                # el 20 % de la gravedad al final del giro
RIG_PDG = KTAU*KP                                        # parado, el PD + g: 1.25·Kp·e
RIG_PAR = (FC*Mq(QF) + JR*np.eye(2))*WN**2               # parado, el par calculado: (0.8·M + 0.25·I)·wn²·e
ERR_PDG = np.degrees(G_FALTA/RIG_PDG)
ERR_PAR = np.degrees(np.linalg.solve(RIG_PAR, G_FALTA))

In [ ]:
# @title ✔ Resultado
print(f'el {100*(1 - FC):.0f} % de g al final:', G_FALTA.round(3), 'N·m')
print(f'PD + g: rigidez del hombro {RIG_PDG[0]:.1f} N·m/rad -> error {ERR_PDG.round(2)}°')
print(f'par calculado: rigidez del hombro {RIG_PAR[0, 0]:.1f} N·m/rad -> error {ERR_PAR.round(2)}°')

**Qué tiene que salir:** el 20 % de g al final, 1,47 N·m en el hombro; rigidez 67,5 N·m/rad con el PD + g y 50,8 con el par calculado: 1,25° y 1,67°, casi los 1,26° y 1,68° de la simulación.

### Los dos juntos: PID + ĝ

El PD + ĝ con el modelo corto y el I de S14 en cada articulación, dentro de su cota (ejercicio 2), con unos segundos más tras el giro.

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# el término I de S14, hombro y codo [V/(rad·s)]
KI = [100.0, 40.0]  # @param {type:"raw"}
# tiempo que se simula tras el giro [s]
DESPUES = 2.0  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
PIDG = {}
for T in T_GIROS:
    _, e, _ = sim2R('pd+g', T, factor=FC, Ki=KI, despues=DESPUES)
    PIDG[T] = (np.abs(e).max(axis=0), e[int(round(T/H))], e[-1])    # máximo, al acabar el giro y al final

In [ ]:
# @title ✔ Resultado
for T, (emax, efin, edes) in PIDG.items():
    print(f'PID + g, modelo x{FC}, giro en {T} s: error máximo {emax.round(3)}°, '
          f'al acabar el giro {efin.round(3)}°, {DESPUES:g} s después {edes.round(3)}°')

**Qué tiene que salir:** dos segundos después del giro de 2 s, el error baja a 0,002°: lo que el modelo no sabe de la gravedad lo quita el I.

### Ejercicio 1 (en casa)

Comprobad, desde `A1`, `A2`, `M1` y `M2`, los coeficientes de M(q) y g(q) (0,285, 0,14, 0,07, 0,04, 0,7 y 0,2), y calculad M, el acoplo sin rotores y con ellos, y g con el codo a 90° y en la postura de S10, (1,0°; 74,5°).

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# posturas, (q1, q2) [°]: codo a 90° y la de S10
E1_POSTURAS = [[0, 90], [1.0, 74.5]]  # @param {type:"raw"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
# M11 = 0.285 + 0.14·cos q2, M12 = 0.04 + 0.07·cos q2, M22 = 0.04; g = 9.81·(0.7·cos q1 + 0.2·cos(q1 + q2); 0.2·cos(q1 + q2))
E1_COEF = [M1*A1**2 + M2*(A1**2 + A2**2), 2*M2*A1*A2, M2*A1*A2, M2*A2**2, (M1 + M2)*A1, M2*A2]
E1_FILAS = []
for grados in E1_POSTURAS:
    e1_q = np.radians(grados); e1_M = Mq(e1_q); e1_Mr = e1_M + JR*np.eye(2)
    E1_FILAS.append((grados, e1_M, e1_M[0, 1]/np.sqrt(e1_M[0, 0]*e1_M[1, 1]),
                     e1_Mr[0, 1]/np.sqrt(e1_Mr[0, 0]*e1_Mr[1, 1]), gq(e1_q)))

In [ ]:
# @title ✔ Resultado
print('coeficientes (0.285, 0.14, 0.07, 0.04, 0.7, 0.2):', [round(x, 4) for x in E1_COEF])
for grados, M, ac, acr, g in E1_FILAS:
    print(f'({grados[0]}°, {grados[1]}°): M = {np.round(M, 3).tolist()}   acoplo {ac:.3f} sin rotores y {acr:.3f} con ellos   g = {np.round(g, 2)} N·m')

### Ejercicio 2 (en casa)

El PID por articulación: el I de S14 sobre el PD, sin compensar la gravedad. ¿Cabe K_i en la cota de S14, (a + bv·K_d)·K_p, con a = 3,125/M_jj y bv = 1,25/M_jj? ¿Qué error queda al acabar el giro y dos segundos después?

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# el término I, hombro y codo [V/(rad·s)]
E2_KI = [100.0, 40.0]  # @param {type:"raw"}
# tiempo que se simula tras el giro [s]
E2_DESPUES = 2.0  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
E2_COTA = (BEQ/Mjj + KTAU/Mjj*KD)*KP                    # la cota de S14, (a + bv·Kd)·Kp
E2_SIMS = {}
for T in T_GIROS:
    _, e, _ = sim2R('pd', T, Ki=E2_KI, despues=E2_DESPUES)
    E2_SIMS[T] = (np.abs(e).max(axis=0), e[int(round(T/H))], e[-1])  # máximo, al acabar el giro y al final

In [ ]:
# @title ✔ Resultado
print('cotas de Ki:', E2_COTA.round(1), 'V/(rad·s)   Ki =', E2_KI)
for T, (emax, efin, edes) in E2_SIMS.items():
    print(f'PID, giro en {T} s: error máximo {emax.round(2)}°, al acabar el giro {efin.round(2)}°, '
          f'{E2_DESPUES:g} s después {edes.round(3)}°')

### Ejercicio 3 (en casa)

Con G = 100: ¿cuánto pesa el rotor visto desde la articulación, cuánto varía M₁₁ + rotor entre plegado y estirado, cuánto vale el acoplo estirado y qué velocidad da el motor en la articulación con 24 V, sin carga, 24/(G·K)?

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# la reductora [-]
E3_G = 100  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
E3_JR = E3_G**2*JM                                       # el rotor visto desde la articulación
E3_VARIA = (Mq([0, 0])[0, 0] + E3_JR)/(Mq([0, np.pi])[0, 0] + E3_JR)   # M11 + rotor, estirado / plegado
E3_Mr = Mq([0, 0]) + E3_JR*np.eye(2)
E3_ACOPLO = E3_Mr[0, 1]/np.sqrt(E3_Mr[0, 0]*E3_Mr[1, 1])             # estirado, con los rotores
E3_VEL = V_BUS/(E3_G*K)                                  # rad/s en la articulación, sin carga

In [ ]:
# @title ✔ Resultado
print(f'G = {E3_G}: rotor {E3_JR:.2f} kg·m²; M11 + rotor varía {E3_VARIA:.2f} veces; acoplo estirado {E3_ACOPLO:.3f}; '
      f'con 24 V, {E3_VEL:.1f} rad/s')

---

## Soluciones

**Ejercicio 1.** Salen 0,285, 0,14, 0,07, 0,04, 0,7 y 0,2. Codo a 90°: M = [[0,285; 0,04]; [0,04; 0,04]], acoplo 0,375 sin rotores y 0,102 con ellos, g = (6,87; 0) N·m: la mano está encima del codo. En (1,0°; 74,5°): M = [[0,322; 0,059]; [0,059; 0,04]], acoplo 0,517 y 0,144, g = (7,36; 0,49) N·m. Los rotores hacen M más diagonal en todas las posturas.

**Ejercicio 2.** Las cotas son 20·K_p, 1080 y 464: 100 y 40 caben. En 2 s, 3,67° de máximo en el hombro, 0,91° al acabar el giro y prácticamente 0 dos segundos después; en 0,5 s, 6,05° y 5,34°. El I quita el error final, no el de seguimiento: necesita tiempo para acumular lo que pide la gravedad.

**Ejercicio 3.** El rotor pesa 1,00 kg·m² (0,25 con G = 50); M₁₁ + rotor varía 1,24 veces (1,71) y el acoplo estirado baja a 0,09 (0,25). Pero con 24 V la articulación no pasa de 4,8 rad/s, la mitad: la reductora iguala y desacopla a cambio de velocidad, como en S13.

---

## Para llevarse de esta sesión

- **Euler-Lagrange da M, c y g sin dibujar fuerzas;** M = Σ m·Jᵀ·J, simétrica y definida positiva.
- **La gravedad domina a velocidades normales;** cuatro veces más rápido, la inercia se multiplica por dieciséis.
- **Con reductora, un PD por articulación casi basta,** salvo la gravedad, que lo deja caído.
- **El par calculado cancela el modelo:** error lineal, desacoplado e independiente de la postura. Su precio, el modelo.
- **Con el modelo equivocado, lo que falta de la gravedad lo pone un I.**

*Cuaderno del curso 82514 Mecatrónica y Robótica · IQS Universitat Ramon Llull · curso 2026/27*

*© Guillermo Reyes Carmenaty · Publicado bajo [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es): puedes usarlo, adaptarlo y redistribuirlo, incluso con fines comerciales, siempre que reconozcas la autoría.*